# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [91]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [92]:
df = pd.read_csv("data/AviationData.csv", encoding ="cp1252", low_memory= False)
print(f"Dataset shape: {df.shape[0]:,}rows")
print(f"Dataset shape: {df.shape[1]}columns")
#print(df.shape)
df.shape

Dataset shape: 88,889rows
Dataset shape: 31columns


(88889, 31)

In [93]:
display(df.head())

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [94]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [95]:
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [96]:
df.dtypes

Event.Id                      str
Investigation.Type            str
Accident.Number               str
Event.Date                    str
Location                      str
Country                       str
Latitude                      str
Longitude                     str
Airport.Code                  str
Airport.Name                  str
Injury.Severity               str
Aircraft.damage               str
Aircraft.Category             str
Registration.Number           str
Make                          str
Model                         str
Amateur.Built                 str
Number.of.Engines         float64
Engine.Type                   str
FAR.Description               str
Schedule                      str
Purpose.of.flight             str
Air.carrier                   str
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition             str
Broad.phase.of.flight         str
Report.Status 

In [97]:
# summary of the dataframe, include each column's datatype and how many values are non_missing
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(), # .isna() marks each missing value True and sums them
    "missing_percent": (df.isna().mean() * 100).round(2) # culculates the share of missing values and convert to percentage in two decimal places.
}).sort_values("missing_percent", ascending=False)

display(missing_summary)
display(df.describe(include="all").T) # transpose the result

,missing_count,missing_percent
Schedule,76307,85.85
Air.carrier,72241,81.27
FAR.Description,56866,63.97
Aircraft.Category,56602,63.68
Longitude,54516,61.33
Latitude,54507,61.32
Airport.Code,38757,43.60
Airport.Name,36185,40.71
Broad.phase.of.flight,27165,30.56
Publication.Date,13771,15.49


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Event.Id,88889,87951,20001214X45071,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Investigation.Type,88889,2,Accident,85015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Accident.Number,88889,88863,ERA22LA103,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event.Date,88889,14782,1982-05-16,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,88837,27758,"ANCHORAGE, AK",434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,88663,219,United States,82248,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,34382,25589,332739N,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Longitude,34373,27154,0112457W,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Code,50132,10374,NONE,1488,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Name,52704,24870,Private,240,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [98]:
#clumns to clean or filter
relevant_cols = [
    "Event.Date",
    "Investigation.Type",   
    "Aircraft.Category",
    "Amateur.Built",
    "Make",
    "Model",
    "Aircraft.damage",
    "Number.of.Engines",
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
    "Weather.Condition",
    "Broad.phase.of.flight",
    "Purpose.of.flight"
]

In [99]:
#check that the expected columns are present
missing_cols = [col for col in relevant_cols if col not in df.columns]

In [100]:
if missing_cols:
    raise KeyError(f"Columns not found in the dataset: {missing_cols}")

In [101]:
#review example records and missing values
display(df[relevant_cols].head())

,Event.Date,Investigation.Type,Aircraft.Category,Amateur.Built,Make,Model,Aircraft.damage,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Purpose.of.flight
0,1948-10-24,Accident,NaN,No,Stinson,108-3,Destroyed,1.0,2.0,0.0,0.0,0.0,UNK,Cruise,Personal
1,1962-07-19,Accident,NaN,No,Piper,PA24-180,Destroyed,1.0,4.0,0.0,0.0,0.0,UNK,Unknown,Personal
2,1974-08-30,Accident,NaN,No,Cessna,172M,Destroyed,1.0,3.0,NaN,NaN,NaN,IMC,Cruise,Personal
3,1977-06-19,Accident,NaN,No,Rockwell,112,Destroyed,1.0,2.0,0.0,0.0,0.0,IMC,Cruise,Personal
4,1979-08-02,Accident,NaN,No,Cessna,501,Destroyed,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Personal


In [102]:
display(pd.DataFrame({
    "missing_count": df[relevant_cols].isna().sum(),
    "missing_percent": (df[relevant_cols].isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False))# lists the columns with the most missing data first.


,missing_count,missing_percent
Aircraft.Category,56602,63.68
Broad.phase.of.flight,27165,30.56
Total.Serious.Injuries,12510,14.07
Total.Minor.Injuries,11933,13.42
Total.Fatal.Injuries,11401,12.83
Purpose.of.flight,6192,6.97
Number.of.Engines,6084,6.84
Total.Uninjured,5912,6.65
Weather.Condition,4492,5.05
Aircraft.damage,3194,3.59


In [103]:
# display the most frequent values in each relevant column include the missing values
for col in relevant_cols:
    print(f"\nValues in {col}:")
    display(df[col].value_counts(dropna=False).head(15))



Values in Event.Date:


Event.Date
1982-05-16    25
1984-06-30    25
2000-07-08    25
1983-06-05    24
1983-08-05    24
1984-08-25    24
1986-05-17    24
1982-05-01    23
1982-07-09    23
1982-10-03    23
1983-05-28    23
1988-08-07    23
2001-06-16    23
2001-07-21    23
1982-04-25    22
Name: count, dtype: int64


Values in Investigation.Type:


Investigation.Type
Accident    85015
Incident     3874
Name: count, dtype: int64


Values in Aircraft.Category:


...

Weather.Condition
VMC    77303
IMC     5976
NaN     4492
UNK      856
Unk      262
Name: count, dtype: int64


Values in Broad.phase.of.flight:


Broad.phase.of.flight
NaN            27165
Landing        15428
Takeoff        12493
Cruise         10269
Maneuvering     8144
Approach        6546
Climb           2034
Taxi            1958
Descent         1887
Go-around       1353
Standing         945
Unknown          548
Other            119
Name: count, dtype: int64


Values in Purpose.of.flight:


Purpose.of.flight
Personal               49448
Instructional          10601
Unknown                 6802
NaN                     6192
Aerial Application      4712
Business                4018
Positioning             1646
Other Work Use          1264
Ferry                    812
Aerial Observation       794
Public Aircraft          720
Executive/corporate      553
Flight Test              405
Skydiving                182
External Load            123
Name: count, dtype: int64

In [104]:
#make a clean working copy and standardize text
df_clean = df.copy()


In [105]:
#define text vslues to be treated as missing

missing_labels = {
    "", "unknown", "unk", "n/a", "na", "not reported", "undetermined"
}


In [106]:
#find columns that panda recognizes as text. not including numeric columns

text_cols = df_clean.select_dtypes(include=["object", "string"]).columns


In [107]:
for col in text_cols:
    values = df_clean[col].astype("string").str.strip()
    df_clean[col] = values.mask(values.str.casefold().isin(missing_labels))


In [108]:
for col in ["Make", "Model"]:
    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
    )


In [109]:
#filter to the records
df_clean["Event.Date"] = pd.to_datetime(
    df_clean["Event.Date"],
    errors="coerce"
)

date_ok = df_clean["Event.Date"].ge("1983-01-01").fillna(False)

date_ok

0        False
1        False
2        False
3        False
4        False
         ...  
88884     True
88885     True
88886     True
88887     True
88888     True
Name: Event.Date, Length: 88889, dtype: bool

In [110]:
accident_ok = (
    df_clean["Investigation.Type"].str.casefold().eq("accident").fillna(False)
)
accident_ok

0        True
1        True
2        True
3        True
4        True
         ... 
88884    True
88885    True
88886    True
88887    True
88888    True
Name: Investigation.Type, Length: 88889, dtype: boolean

In [111]:
professional_build_ok = (
    df_clean["Amateur.Built"].str.casefold().eq("no").fillna(False)
)
professional_build_ok

0        True
1        True
2        True
3        True
4        True
         ... 
88884    True
88885    True
88886    True
88887    True
88888    True
Name: Amateur.Built, Length: 88889, dtype: boolean

In [112]:
airplane_ok = (
    df_clean["Aircraft.Category"].str.casefold().eq("airplane").fillna(False)
)
airplane_ok

0        False
1        False
2        False
3        False
4        False
         ...  
88884    False
88885    False
88886     True
88887    False
88888    False
Name: Aircraft.Category, Length: 88889, dtype: boolean

In [113]:
keep_rows = date_ok & accident_ok & professional_build_ok & airplane_ok
keep_rows

0        False
1        False
2        False
3        False
4        False
         ...  
88884    False
88885    False
88886     True
88887    False
88888    False
Length: 88889, dtype: boolean

In [114]:
print(f"Rows before filtering: {len(df_clean):,}")
print(f"Rows meeting all criteria: {keep_rows.sum():,}")

df_clean = df_clean.loc[keep_rows].copy()
#df_clean

Rows before filtering: 88,889
Rows meeting all criteria: 19,937


In [115]:
print("Filtered date range:", df_clean["Event.Date"].min(),
      "to", df_clean["Event.Date"].max())


Filtered date range: 1983-03-20 00:00:00 to 2022-12-26 00:00:00


In [116]:
#create occupant and injury measures
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

In [117]:
# Make sure these columns contain numbers
df_clean[injury_cols] = df_clean[injury_cols].apply(
    pd.to_numeric, errors="coerce"
)
df_clean[injury_cols]

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
4171,1.0,1.0,NaN,NaN
4285,1.0,NaN,NaN,4.0
5960,11.0,2.0,NaN,13.0
6669,1.0,NaN,NaN,29.0
6806,NaN,NaN,NaN,1.0
...,...,...,...,...
88869,0.0,0.0,0.0,1.0
88873,0.0,0.0,0.0,1.0
88876,0.0,0.0,0.0,1.0
88877,0.0,1.0,0.0,0.0


In [118]:
# Add the injury counts to estimate the total people on board.
# min_count=4 means all four counts must be present.
df_clean["Total.Occupants"] = df_clean[injury_cols].sum(
    axis=1, min_count=4
)
df_clean["Total.Occupants"]

4171     NaN
4285     NaN
5960     NaN
6669     NaN
6806     NaN
        ... 
88869    1.0
88873    1.0
88876    1.0
88877    1.0
88886    1.0
Name: Total.Occupants, Length: 19937, dtype: float64

In [119]:
# Add fatal and serious injuries
df_clean["Severe.Fatal.Count"] = (
    df_clean["Total.Fatal.Injuries"]
    + df_clean["Total.Serious.Injuries"]
)
df_clean["Severe.Fatal.Count"]

4171      2.0
4285      NaN
5960     13.0
6669      NaN
6806      NaN
         ... 
88869     0.0
88873     0.0
88876     0.0
88877     1.0
88886     0.0
Name: Severe.Fatal.Count, Length: 19937, dtype: float64

In [120]:
# Calculate the fraction of occupants with fatal or serious injuries
df_clean["Severe.Fatal.Fraction"] = (
    df_clean["Severe.Fatal.Count"] / df_clean["Total.Occupants"]
)
df_clean["Severe.Fatal.Count"]

4171      2.0
4285      NaN
5960     13.0
6669      NaN
6806      NaN
         ... 
88869     0.0
88873     0.0
88876     0.0
88877     1.0
88886     0.0
Name: Severe.Fatal.Count, Length: 19937, dtype: float64

In [121]:
# Do not calculate a fraction for missing or zero occupant totals
df_clean.loc[
    df_clean["Total.Occupants"].isna()
    | (df_clean["Total.Occupants"] == 0),
    "Severe.Fatal.Fraction"
] = np.nan


In [122]:
# Group flights by the number of people recorded on board
df_clean["Aircraft.Size"] = pd.NA
df_clean.loc[
    df_clean["Total.Occupants"].between(1, 20),
    "Aircraft.Size"
] = "Small"
df_clean.loc[
    df_clean["Total.Occupants"] > 20,
    "Aircraft.Size"
] = "Large"

In [123]:
# Create a destruction indicator only when damage status is reported.
df_clean["Aircraft.Destroyed"] = (
    df_clean["Aircraft.damage"]
    .astype("string")
    .str.casefold()
    .eq("destroyed")
    .astype("Int64")
)
df_clean.loc[df_clean["Aircraft.damage"].isna(), "Aircraft.Destroyed"] = pd.NA

In [124]:
# Preserve missingness flags and make Unknown available for descriptive summaries.
descriptive_cols = [
    "Weather.Condition",
    "Broad.phase.of.flight",
    "Purpose.of.flight"
]

for col in descriptive_cols:
    df_clean[f"{col}_was_missing"] = df_clean[col].isna()
    df_clean[f"{col}_for_summary"] = df_clean[col].fillna("Unknown")

In [125]:
# Optional labels for general summaries; use original Make/Model for rankings.
df_clean["Make_for_summary"] = df_clean["Make"].fillna("Unknown")
df_clean["Model_for_summary"] = df_clean["Model"].fillna("Unknown")

In [126]:
 #review and save the cleaned data
print(f"Filtered accident records: {len(df_clean):,}")
print("Records with complete occupant counts:",
      f"{df_clean['Total.Occupants'].notna().sum():,}")
print("Records with known damage:",
      f"{df_clean['Aircraft.Destroyed'].notna().sum():,}")

display(df_clean["Aircraft.Size"].value_counts(dropna=False))
display(df_clean.head())

df_clean.to_csv("data/AviationData_cleaned.csv", index=False)
print("Saved cleaned data to data/AviationData_cleaned.csv")


Filtered accident records: 19,937
Records with complete occupant counts: 17,015
Records with known damage: 19,428


Aircraft.Size
Small    16282
<NA>      3167
Large      488
Name: count, dtype: int64

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Aircraft.Size,Aircraft.Destroyed,Weather.Condition_was_missing,Weather.Condition_for_summary,Broad.phase.of.flight_was_missing,Broad.phase.of.flight_for_summary,Purpose.of.flight_was_missing,Purpose.of.flight_for_summary,Make_for_summary,Model_for_summary
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,<NA>,<NA>,<NA>,<NA>,...,<NA>,1,False,IMC,False,Cruise,False,Personal,PIPER,PA-28-140
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,<NA>,<NA>,TX05,AERO COUNTRY,...,<NA>,<NA>,False,VMC,False,Standing,False,Skydiving,DE HAVILLAND,DHC-6
5960,20001214X44100,Accident,DCA83AA036,1983-08-21,"SILVANA, WA",United States,<NA>,<NA>,S88,<NA>,...,<NA>,1,False,VMC,False,Other,False,Skydiving,LOCKHEED,"LEARSTAR, L-18-56"
6669,20001214X44944,Accident,NYC84LA015,1983-10-28,"MIDDLETOWN, PA",United States,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,False,VMC,False,Climb,True,Unknown,SHORT BROTHERS,SD3-30
6806,20001214X45188,Accident,NYC84LA028,1983-11-13,"MARTHA'S VINEYARD, MA",United States,<NA>,<NA>,<NA>,<NA>,...,<NA>,0,False,VMC,False,Climb,False,Personal,BEECH,C35


Saved cleaned data to data/AviationData_cleaned.csv


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

The four injury counts are used to estimate the total number of people aboard. Missing counts are left unknown, not treated as zero, because zero injuries and unreported injuries are different.

In [127]:
# These four columns are used to estimate the total people aboard each flight.
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

In [128]:
# Convert the counts to numbers. Values that cannot be converted become missing.
for col in injury_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Estimate the total people aboard only when all four injury counts are present.
df_clean["Total.Occupants"] = df_clean[injury_cols].sum(
    axis=1,
    min_count=4
)

In [129]:
# Add fatal and serious injuries.
df_clean["Fatal.Serious.Injuries"] = (
    df_clean["Total.Fatal.Injuries"]
    + df_clean["Total.Serious.Injuries"]
)

In [130]:
# Calculate the fraction of occupants with a fatal or serious injury.
# Do not calculate it when the occupant count is missing or zero.
df_clean["Fatal.Serious.Injury.Fraction"] = (
    df_clean["Fatal.Serious.Injuries"] / df_clean["Total.Occupants"]
)

df_clean.loc[
    df_clean["Total.Occupants"].isna()
    | (df_clean["Total.Occupants"] == 0),
    "Fatal.Serious.Injury.Fraction"
] = np.nan

In [131]:
# Optional: express the fraction as a percentage.
df_clean["Fatal.Serious.Injury.Percent"] = (
    df_clean["Fatal.Serious.Injury.Fraction"] * 100
)

In [132]:
# Check the new measures.
display(df_clean[[
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
    "Total.Occupants",
    "Fatal.Serious.Injuries",
    "Fatal.Serious.Injury.Fraction",
    "Fatal.Serious.Injury.Percent"
]].head())

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Total.Occupants,Fatal.Serious.Injuries,Fatal.Serious.Injury.Fraction,Fatal.Serious.Injury.Percent
4171,1.0,1.0,NaN,NaN,NaN,2.0,NaN,NaN
4285,1.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN
5960,11.0,2.0,NaN,13.0,NaN,13.0,NaN,NaN
6669,1.0,NaN,NaN,29.0,NaN,NaN,NaN,NaN
6806,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [133]:
# Inspect the damage values before recoding them
display(df_clean["Aircraft.damage"].value_counts(dropna=False))

Aircraft.damage
Substantial    16965
Destroyed       2316
<NA>             509
Minor            147
Name: count, dtype: Int64

In [134]:
# Clean spaces and make the values lowercase for consistent comparison
damage = (
    df_clean["Aircraft.damage"]
    .astype("string")
    .str.strip()
    .str.casefold()
)


In [135]:
# Treat blank and unknown labels as missing
damage = damage.mask(
    damage.isin(["", "unknown", "unk", "n/a", "na", "undetermined"])
)


In [136]:
# Create a nullable indicator:
# 1 = destroyed, 0 = known damage but not destroyed, missing = unknown/not reported
df_clean["Aircraft.Destroyed"] = pd.Series(
    pd.NA, index=df_clean.index, dtype="Int64"
)

df_clean.loc[damage.eq("destroyed").fillna(False), "Aircraft.Destroyed"] = 1
df_clean.loc[
    damage.isin(["substantial", "minor", "none"]).fillna(False),
    "Aircraft.Destroyed"
] = 0

In [137]:
# Check the result
display(df_clean[["Aircraft.damage", "Aircraft.Destroyed"]].head(10))
display(df_clean["Aircraft.Destroyed"].value_counts(dropna=False))

,Aircraft.damage,Aircraft.Destroyed
4171,Destroyed,1
4285,<NA>,<NA>
5960,Destroyed,1
6669,<NA>,<NA>
6806,Substantial,0
7084,Substantial,0
7708,Substantial,0
8585,Substantial,0
8591,Substantial,0
8865,Destroyed,1


Aircraft.Destroyed
0       17112
1        2316
<NA>      509
Name: count, dtype: Int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [138]:
# Check the original Make values, including missing values
display(df_clean["Make"].value_counts(dropna=False).head(20))

Make
CESSNA                7029
PIPER                 3938
BEECH                 1392
BOEING                 553
MOONEY                 358
BELLANCA               217
AIR TRACTOR INC        217
MAULE                  215
CIRRUS DESIGN CORP     209
AIR TRACTOR            206
AERONCA                200
CHAMPION               157
GRUMMAN                146
LUSCOMBE               141
CIRRUS                 130
STINSON                129
NORTH AMERICAN         104
EMBRAER                 98
TAYLORCRAFT             93
DEHAVILLAND             93
Name: count, dtype: Int64

In [139]:
# Standardize spacing and capitalization
df_clean["Make_clean"] = (
    df_clean["Make"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)
df_clean["Make_clean"]

4171                          PIPER
4285                   DE HAVILLAND
5960                       LOCKHEED
6669                 SHORT BROTHERS
6806                          BEECH
                    ...            
88869                         PIPER
88873            CIRRUS DESIGN CORP
88876                    SWEARINGEN
88877                        CESSNA
88886    AMERICAN CHAMPION AIRCRAFT
Name: Make_clean, Length: 19937, dtype: string

In [140]:
# Treat blank and common unknown labels as missing
unknown_labels = ["", "UNKNOWN", "UNK", "N/A", "NA", "NOT REPORTED"]

df_clean["Make_clean"] = df_clean["Make_clean"].replace(
    unknown_labels, pd.NA
)
unknown_labels

['', 'UNKNOWN', 'UNK', 'N/A', 'NA', 'NOT REPORTED']

In [141]:
# Count accident records for each known Make
make_counts = (
    df_clean["Make_clean"]
    .value_counts()
    .rename_axis("Make")
    .reset_index(name="Accident_Count")
)


In [142]:
# Keep Makes with at least 50 records
eligible_makes = make_counts.loc[
    make_counts["Accident_Count"] >= 50, "Make"
]


In [143]:
# Use this dataframe for Make-level comparisons
df_make_analysis = df_clean[
    df_clean["Make_clean"].isin(eligible_makes)
].copy()

print(f"Makes with at least 50 accident records: {len(eligible_makes)}")
print(f"Records retained for Make analysis: {len(df_make_analysis):,}")

display(make_counts.head(20))

Makes with at least 50 accident records: 34
Records retained for Make analysis: 16,550


,Make,Accident_Count
0,CESSNA,7029
1,PIPER,3938
2,BEECH,1392
3,BOEING,553
4,MOONEY,358
5,BELLANCA,217
6,AIR TRACTOR INC,217
7,MAULE,215
8,CIRRUS DESIGN CORP,209
9,AIR TRACTOR,206


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [144]:
# Make a separate dataframe for the Make/Model analysis
model_df = df_clean.copy()
model_df

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Broad.phase.of.flight_was_missing,Broad.phase.of.flight_for_summary,Purpose.of.flight_was_missing,Purpose.of.flight_for_summary,Make_for_summary,Model_for_summary,Fatal.Serious.Injuries,Fatal.Serious.Injury.Fraction,Fatal.Serious.Injury.Percent,Make_clean
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,<NA>,<NA>,<NA>,<NA>,...,False,Cruise,False,Personal,PIPER,PA-28-140,2.0,NaN,NaN,PIPER
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,<NA>,<NA>,TX05,AERO COUNTRY,...,False,Standing,False,Skydiving,DE HAVILLAND,DHC-6,NaN,NaN,NaN,DE HAVILLAND
5960,20001214X44100,Accident,DCA83AA036,1983-08-21,"SILVANA, WA",United States,<NA>,<NA>,S88,<NA>,...,False,Other,False,Skydiving,LOCKHEED,"LEARSTAR, L-18-56",13.0,NaN,NaN,LOCKHEED
6669,20001214X44944,Accident,NYC84LA015,1983-10-28,"MIDDLETOWN, PA",United States,<NA>,<NA>,<NA>,<NA>,...,False,Climb,True,Unknown,SHORT BROTHERS,SD3-30,NaN,NaN,NaN,SHORT BROTHERS
6806,20001214X45188,Accident,NYC84LA028,1983-11-13,"MARTHA'S VINEYARD, MA",United States,<NA>,<NA>,<NA>,<NA>,...,False,Climb,False,Personal,BEECH,C35,NaN,NaN,NaN,BEECH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88869,20221213106455,Accident,WPR23LA065,2022-12-13,"Lewistown, MT",United States,047257N,0109280W,KLWT,Lewiston Municipal Airport,...,True,Unknown,True,Unknown,PIPER,PA42,0.0,0.0,0.0,PIPER
88873,20221215106463,Accident,ERA23LA090,2022-12-14,"San Juan, PR",United States,182724N,0066554W,SIG,FERNANDO LUIS RIBAS DOMINICCI,...,True,Unknown,False,Personal,CIRRUS DESIGN CORP,SR22,0.0,0.0,0.0,CIRRUS DESIGN CORP
88876,20221219106475,Accident,WPR23LA069,2022-12-15,"Wichita, KS",United States,373829N,0972635W,ICT,WICHITA DWIGHT D EISENHOWER NT,...,True,Unknown,True,Unknown,SWEARINGEN,SA226TC,0.0,0.0,0.0,SWEARINGEN
88877,20221219106470,Accident,ERA23LA091,2022-12-16,"Brooksville, FL",United States,282825N,0822719W,BKV,BROOKSVILLE-TAMPA BAY RGNL,...,True,Unknown,False,Personal,CESSNA,R172K,1.0,1.0,100.0,CESSNA


In [145]:
# Standardize Model labels
model_df["Model_clean"] = (
    model_df["Model"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)
model_df["Make_clean"]

4171                          PIPER
4285                   DE HAVILLAND
5960                       LOCKHEED
6669                 SHORT BROTHERS
6806                          BEECH
                    ...            
88869                         PIPER
88873            CIRRUS DESIGN CORP
88876                    SWEARINGEN
88877                        CESSNA
88886    AMERICAN CHAMPION AIRCRAFT
Name: Make_clean, Length: 19937, dtype: string

In [146]:
# Treat blank and common unknown labels as missing
unknown_labels = ["", "UNKNOWN", "UNK", "N/A", "NA", "NOT REPORTED"]
model_df["Model_clean"] = model_df["Model_clean"].replace(
    unknown_labels, pd.NA
)
unknown_labels

['', 'UNKNOWN', 'UNK', 'N/A', 'NA', 'NOT REPORTED']

In [147]:
# Remove rows without a Make or Model from this analysis dataframe
model_df = model_df.dropna(
    subset=["Make_clean", "Model_clean"]
).copy()
model_df

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Broad.phase.of.flight_for_summary,Purpose.of.flight_was_missing,Purpose.of.flight_for_summary,Make_for_summary,Model_for_summary,Fatal.Serious.Injuries,Fatal.Serious.Injury.Fraction,Fatal.Serious.Injury.Percent,Make_clean,Model_clean
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,<NA>,<NA>,<NA>,<NA>,...,Cruise,False,Personal,PIPER,PA-28-140,2.0,NaN,NaN,PIPER,PA-28-140
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,<NA>,<NA>,TX05,AERO COUNTRY,...,Standing,False,Skydiving,DE HAVILLAND,DHC-6,NaN,NaN,NaN,DE HAVILLAND,DHC-6
5960,20001214X44100,Accident,DCA83AA036,1983-08-21,"SILVANA, WA",United States,<NA>,<NA>,S88,<NA>,...,Other,False,Skydiving,LOCKHEED,"LEARSTAR, L-18-56",13.0,NaN,NaN,LOCKHEED,"LEARSTAR, L-18-56"
6669,20001214X44944,Accident,NYC84LA015,1983-10-28,"MIDDLETOWN, PA",United States,<NA>,<NA>,<NA>,<NA>,...,Climb,True,Unknown,SHORT BROTHERS,SD3-30,NaN,NaN,NaN,SHORT BROTHERS,SD3-30
6806,20001214X45188,Accident,NYC84LA028,1983-11-13,"MARTHA'S VINEYARD, MA",United States,<NA>,<NA>,<NA>,<NA>,...,Climb,False,Personal,BEECH,C35,NaN,NaN,NaN,BEECH,C35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88869,20221213106455,Accident,WPR23LA065,2022-12-13,"Lewistown, MT",United States,047257N,0109280W,KLWT,Lewiston Municipal Airport,...,Unknown,True,Unknown,PIPER,PA42,0.0,0.0,0.0,PIPER,PA42
88873,20221215106463,Accident,ERA23LA090,2022-12-14,"San Juan, PR",United States,182724N,0066554W,SIG,FERNANDO LUIS RIBAS DOMINICCI,...,Unknown,False,Personal,CIRRUS DESIGN CORP,SR22,0.0,0.0,0.0,CIRRUS DESIGN CORP,SR22
88876,20221219106475,Accident,WPR23LA069,2022-12-15,"Wichita, KS",United States,373829N,0972635W,ICT,WICHITA DWIGHT D EISENHOWER NT,...,Unknown,True,Unknown,SWEARINGEN,SA226TC,0.0,0.0,0.0,SWEARINGEN,SA226TC
88877,20221219106470,Accident,ERA23LA091,2022-12-16,"Brooksville, FL",United States,282825N,0822719W,BKV,BROOKSVILLE-TAMPA BAY RGNL,...,Unknown,False,Personal,CESSNA,R172K,1.0,1.0,100.0,CESSNA,R172K


In [148]:
# Count accident records for each Make/Model pair
model_make_counts = (
    model_df.groupby(["Make_clean", "Model_clean"])
    .size()
    .reset_index(name="Accident_Count")
    .sort_values("Accident_Count", ascending=False)
)


In [149]:
print("Most frequent Make/Model combinations:")
display(model_make_counts.head(25))

Most frequent Make/Model combinations:


,Make_clean,Model_clean,Accident_Count
1109,CESSNA,172,763
1099,CESSNA,152,311
1172,CESSNA,182,300
1145,CESSNA,172S,272
3277,PIPER,PA28,268
1140,CESSNA,172N,249
1157,CESSNA,180,212
1139,CESSNA,172M,179
1084,CESSNA,150,177
3125,PIPER,PA-18-150,175


In [150]:
# Check whether any Model label is used by more than one Make
makes_per_model = (
    model_df.groupby("Model_clean")["Make_clean"]
    .nunique()
    .sort_values(ascending=False)
)
makes_per_model

Model_clean
CHALLENGER II         14
G-164B                10
G-164A                10
S2R                    9
A36                    9
                      ..
ZENITH CH 701          1
ZENITH CH-750          1
ZENITH STOL CH 701     1
ZEPHYR II              1
108 3                  1
Name: Make_clean, Length: 3240, dtype: int64

In [151]:
shared_model_labels = makes_per_model[makes_per_model > 1]

print(f"Model labels used by more than one Make: {len(shared_model_labels)}")
display(shared_model_labels.head(25))

Model labels used by more than one Make: 507


Model_clean
CHALLENGER II    14
G-164B           10
G-164A           10
S2R               9
A36               9
AA-5A             7
AA-5B             7
DRAGONFLY         7
LC41-550FG        7
7GCBC             7
390               7
S2R-T34           6
G-164             6
CH 2000           6
SPORTSTAR         6
S-2R              6
LC42-550FG        5
CL-600-2B16       5
DA 20-C1          5
CHALLENGER        5
114               5
SR22              5
SR-22             5
SR20              5
PITTS S-2B        5
Name: Make_clean, dtype: int64

In [152]:
# Combine Make and Model to identify each plane type uniquely
model_df["Plane.Type"] = (
    model_df["Make_clean"] + " | " + model_df["Model_clean"]
)
model_df["Plane.Type"]


4171                      PIPER | PA-28-140
4285                   DE HAVILLAND | DHC-6
5960           LOCKHEED | LEARSTAR, L-18-56
6669                SHORT BROTHERS | SD3-30
6806                            BEECH | C35
                        ...                
88869                          PIPER | PA42
88873             CIRRUS DESIGN CORP | SR22
88876                  SWEARINGEN | SA226TC
88877                        CESSNA | R172K
88886    AMERICAN CHAMPION AIRCRAFT | 8GCBC
Name: Plane.Type, Length: 19922, dtype: string

In [153]:
# Count accident records for each unique plane type
plane_type_counts = (
    model_df["Plane.Type"]
    .value_counts()
    .rename_axis("Plane.Type")
    .reset_index(name="Accident_Count")
)
plane_type_counts

,Plane.Type,Accident_Count
0,CESSNA | 172,763
1,CESSNA | 152,311
2,CESSNA | 182,300
3,CESSNA | 172S,272
4,PIPER | PA28,268
...,...,...
4053,CESSNA | P206D,1
4054,PHANTOM | PHANTOM,1
4055,JAMES R DERNOVSEK | KITFOX S5,1
4056,ORLICAN S R O | M-8 EAGLE,1


In [154]:
display(plane_type_counts.head(25))

,Plane.Type,Accident_Count
0,CESSNA | 172,763
1,CESSNA | 152,311
2,CESSNA | 182,300
3,CESSNA | 172S,272
4,PIPER | PA28,268
5,CESSNA | 172N,249
6,CESSNA | 180,212
7,CESSNA | 172M,179
8,CESSNA | 150,177
9,PIPER | PA-18-150,175


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [155]:
other_cols = [
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

In [156]:
# Check that the expected columns are present
missing_cols = [col for col in other_cols if col not in df_clean.columns]

if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}")

In [157]:
# Inspect values and missing counts before cleaning
for col in other_cols:
    print(f"\n{col}:")
    print("Missing values:", df_clean[col].isna().sum())
    display(df_clean[col].value_counts(dropna=False).head(20))



Engine.Type:
Missing values: 3201


Engine.Type
Reciprocating    14893
<NA>              3201
Turbo Prop        1168
Turbo Fan          559
Turbo Jet          101
Turbo Shaft         10
Electric             5
Name: count, dtype: Int64


Weather.Condition:
Missing values: 2175


Weather.Condition
VMC     16734
<NA>     2175
IMC      1028
Name: count, dtype: Int64


Number.of.Engines:
Missing values: 1815


Number.of.Engines
1.0    15624
2.0     2420
NaN     1815
4.0       44
3.0       24
0.0        8
8.0        1
6.0        1
Name: count, dtype: int64


Purpose.of.flight:
Missing values: 2738


Purpose.of.flight
Personal                     11636
<NA>                          2738
Instructional                 2719
Aerial Application             931
Business                       474
Positioning                    331
Skydiving                      163
Aerial Observation             160
Other Work Use                 152
Flight Test                    115
Ferry                          100
Executive/corporate             93
Banner Tow                      89
Public Aircraft - Federal       51
Air Race show                   47
Public Aircraft                 37
Glider Tow                      35
Public Aircraft - State         23
Firefighting                    17
Public Aircraft - Local         11
Name: count, dtype: Int64


Broad.phase.of.flight:
Missing values: 17172


Broad.phase.of.flight
<NA>           17172
Landing         1250
Takeoff          501
Cruise           256
Approach         231
Maneuvering      167
Taxi             104
Go-around         89
Descent           68
Climb             54
Standing          42
Other              3
Name: count, dtype: Int64

In [158]:
# Standardize the text categories
category_cols = [
    "Engine.Type",
    "Weather.Condition",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]
category_cols


['Engine.Type',
 'Weather.Condition',
 'Purpose.of.flight',
 'Broad.phase.of.flight']

In [159]:
for col in category_cols:
    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
        .replace({"UNK": "UNKNOWN", "UNKN": "UNKNOWN"})
    )


In [160]:
# Convert engine count to numeric.
# Values that cannot be converted become missing; rows are not removed.
df_clean["Number.of.Engines"] = pd.to_numeric(
    df_clean["Number.of.Engines"],
    errors="coerce"
)
df_clean["Number.of.Engines"]

4171     1.0
4285     2.0
5960     2.0
6669     2.0
6806     1.0
        ... 
88869    2.0
88873    1.0
88876    2.0
88877    1.0
88886    1.0
Name: Number.of.Engines, Length: 19937, dtype: float64

In [161]:
# Review the cleaned values
for col in other_cols:
    print(f"\nCleaned values in {col}:")
    display(df_clean[col].value_counts(dropna=False).head(20))


Cleaned values in Engine.Type:


Engine.Type
RECIPROCATING    14893
<NA>              3201
TURBO PROP        1168
TURBO FAN          559
TURBO JET          101
TURBO SHAFT         10
ELECTRIC             5
Name: count, dtype: Int64


Cleaned values in Weather.Condition:


Weather.Condition
VMC     16734
<NA>     2175
IMC      1028
Name: count, dtype: Int64


Cleaned values in Number.of.Engines:


Number.of.Engines
1.0    15624
2.0     2420
NaN     1815
4.0       44
3.0       24
0.0        8
8.0        1
6.0        1
Name: count, dtype: int64


Cleaned values in Purpose.of.flight:


Purpose.of.flight
PERSONAL                     11636
<NA>                          2738
INSTRUCTIONAL                 2719
AERIAL APPLICATION             931
BUSINESS                       474
POSITIONING                    331
SKYDIVING                      163
AERIAL OBSERVATION             160
OTHER WORK USE                 152
FLIGHT TEST                    115
FERRY                          100
EXECUTIVE/CORPORATE             93
BANNER TOW                      89
PUBLIC AIRCRAFT - FEDERAL       51
AIR RACE SHOW                   47
PUBLIC AIRCRAFT                 37
GLIDER TOW                      35
PUBLIC AIRCRAFT - STATE         23
FIREFIGHTING                    17
PUBLIC AIRCRAFT - LOCAL         11
Name: count, dtype: Int64


Cleaned values in Broad.phase.of.flight:


Broad.phase.of.flight
<NA>           17172
LANDING         1250
TAKEOFF          501
CRUISE           256
APPROACH         231
MANEUVERING      167
TAXI             104
GO-AROUND         89
DESCENT           68
CLIMB             54
STANDING          42
OTHER              3
Name: count, dtype: Int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [162]:
# Show the percentage of missing values in each column
missing_percent = (df_clean.isna().mean() * 100).sort_values(ascending=False)

display(missing_percent.to_frame("Missing_Percent").round(2))

missing_percent

,Missing_Percent
Schedule,91.57
Broad.phase.of.flight,86.13
Air.carrier,54.01
Airport.Code,32.97
Airport.Name,32.67
Report.Status,18.58
Engine.Type,16.06
Fatal.Serious.Injury.Fraction,15.89
Severe.Fatal.Fraction,15.89
Aircraft.Size,15.89


Schedule                             91.573456
Broad.phase.of.flight                86.131314
Air.carrier                          54.010132
Airport.Code                         32.968852
Airport.Name                         32.672920
Report.Status                        18.583538
Engine.Type                          16.055575
Fatal.Serious.Injury.Fraction        15.885038
Severe.Fatal.Fraction                15.885038
Aircraft.Size                        15.885038
Fatal.Serious.Injury.Percent         15.885038
Total.Occupants                      14.656167
Severe.Fatal.Count                   14.420424
Fatal.Serious.Injuries               14.420424
Purpose.of.flight                    13.733260
Total.Serious.Injuries               13.653007
Total.Fatal.Injuries                 13.261775
Total.Minor.Injuries                 12.248583
Weather.Condition                    10.909364
Number.of.Engines                     9.103677
Longitude                             6.866630
Latitude     

In [163]:
# Keep columns needed for the assignment
important_cols = [
    "Event.Date",
    "Investigation.Type",
    "Aircraft.Category",
    "Amateur.Built",
    "Make",
    "Model",
    "Aircraft.damage",
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight",
    "Total.Occupants",
    "Severe.Fatal.Count",
    "Severe.Fatal.Fraction",
    "Aircraft.Destroyed"
]


In [164]:
# Find columns that are more than 70% missing,
# but do not remove the important analysis columns above
columns_to_drop = [
    col for col in df_clean.columns
    if missing_percent[col] > 70 and col not in important_cols
]

columns_to_drop

['Schedule']

In [165]:
print("Columns to drop:", columns_to_drop)

Columns to drop: ['Schedule']


In [166]:
# Drop those columns
df_clean = df_clean.drop(columns=columns_to_drop)

print("New dataframe shape:", df_clean.shape)

New dataframe shape: (19937, 47)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [167]:
df_clean.to_csv("data/AviationData_cleaned.csv", index=False)

print("Cleaned data saved to data/AviationData_cleaned.csv")


Cleaned data saved to data/AviationData_cleaned.csv
